In [ ]:
pip uninstall -y bitsandbytes

Found existing installation: bitsandbytes 0.48.2
Uninstalling bitsandbytes-0.48.2:
  Successfully uninstalled bitsandbytes-0.48.2


In [ ]:
!pip install -U bitsandbytes

In [ ]:
import bitsandbytes
print(bitsandbytes.__version__)

0.48.2


In [1]:
!apt install -y libomp-dev
!pip install faiss-cpu
#!pip install bitsandbytes

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libllvm14 libomp-14-dev libomp5-14
Suggested packages:
  libomp-14-doc
The following NEW packages will be installed:
  libllvm14 libomp-14-dev libomp-dev libomp5-14
0 upgraded, 4 newly installed, 0 to remove and 41 not upgraded.
Need to get 24.7 MB of archives.
After this operation, 118 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libllvm14 amd64 1:14.0.0-1ubuntu1.1 [24.0 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libomp5-14 amd64 1:14.0.0-1ubuntu1.1 [389 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libomp-14-dev amd64 1:14.0.0-1ubuntu1.1 [347 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libomp-dev amd64 1:14.0-55~exp2 [3,074 B]
Fetched 24.7 MB in 1s (18.3 MB/s)
Selecting previously unselected package l

In [3]:
import os
import pandas as pd
import torch
import traceback
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import gradio as gr
import gc

class MedicalAIFull:
    def __init__(self):
        self.df = None
        self.tokenizer = None
        self.model = None
        self.knowledge_base = []
        self.embedder = None
        self.index = None
        self.system_ready = False
        self.model_loaded = False
        self.rag_ready = False
        self.generator = None
        print("🚀 Medical AI Full-Pipeline Initialized")

    def load_and_clean_dataset(self, file_path: str):
        if not file_path:
            return "❌ Please upload a CSV file!", None

        try:
            df = pd.read_csv(file_path)
            required_cols = ["disease", "symptoms"]
            missing_cols = [c for c in required_cols if c not in df.columns]
            if missing_cols:
                return f"Missing required columns: {', '.join(missing_cols)}", None

            df = df.dropna(subset=required_cols).drop_duplicates(subset=required_cols)
            df = df.reset_index(drop=True)

            df['symptoms'] = df['symptoms'].astype(str).str.strip()
            df['disease'] = df['disease'].astype(str).str.strip()

            self.df = df

            info = (
                f"✅ Dataset loaded & cleaned!\n"
                f"📊 Shape: {self.df.shape}\n"
                f"📝 Columns: {', '.join(self.df.columns)}\n"
                f"📋 Sample size: {len(self.df)} records"
            )
            preview = self.df.head(5)
            return info, preview

        except Exception as e:
            return f"❌ Error loading file: {str(e)}", None

    def setup_model(self,
                    base_model_name="microsoft/DialoGPT-medium",
                    hf_token: str = None):

        print(f"🧠 Setting up model: {base_model_name}")

        try:
            tokenizer_kwargs = {"trust_remote_code": True}
            if hf_token:
                tokenizer_kwargs["use_auth_token"] = hf_token

            self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, **tokenizer_kwargs)
            torch.cuda.empty_cache() if torch.cuda.is_available() else None

            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            # SIMPLE model loading - NO quantization
            model_kwargs = {
                "trust_remote_code": True,
            }

            # Use CPU if no GPU, else use GPU with float16 for efficiency
            if torch.cuda.is_available():
                model_kwargs["torch_dtype"] = torch.float16
                model_kwargs["device_map"] = "auto"
            else:
                model_kwargs["torch_dtype"] = torch.float32
                print("⚠️ Using CPU mode - responses may be slower")

            if hf_token:
                model_kwargs["use_auth_token"] = hf_token

            self.model = AutoModelForCausalLM.from_pretrained(base_model_name, **model_kwargs)

            # Move to device explicitly if not using device_map
            if "device_map" not in model_kwargs:
                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                self.model = self.model.to(device)

            self.model_loaded = True
            print("✅ Model successfully loaded")
            return "✅ Model successfully loaded"

        except Exception as e:
            print(f"Error loading model: {e}")
            self.tokenizer = None
            self.model = None
            return f"❌ Error loading model: {e}"

    def setup_rag(self):
        if self.df is None:
            return "❌ No dataset loaded."

        print("📚 Building optimized RAG...")
        try:
            # Use CPU for embedding to avoid conflicts
            self.embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

            # Create better knowledge base
            self.knowledge_base = []
            for _, row in self.df.iterrows():
                entry = {
                    "disease": row['disease'],
                    "symptoms": row['symptoms'],
                    "text": f"Disease: {row['disease']}. Symptoms: {row['symptoms']}." +
                           (f" Treatment: {row['treatment']}." if "treatment" in self.df.columns and not pd.isna(row.get("treatment","")) else "")
                }
                self.knowledge_base.append(entry)

            # Encode all texts
            texts = [item["text"] for item in self.knowledge_base]
            embeddings = self.embedder.encode(
                texts,
                batch_size=32,
                show_progress_bar=False,
                convert_to_numpy=True
            )

            dim = embeddings.shape[1]
            self.index = faiss.IndexFlatIP(dim)
            faiss.normalize_L2(embeddings)
            self.index.add(embeddings)

            self.rag_ready = True
            return f"✅ RAG ready with {len(self.knowledge_base)} entries"

        except Exception as e:
            print(f"Error loading knowledge base: {e}")
            self.knowledge_base = None
            return f"❌ RAG setup failed: {e}"

    def smart_retrieve(self, query, top_k=3):
        if not self.rag_ready:
            return ""
        try:
            query_embedding = self.embedder.encode([query], convert_to_numpy=True)
            faiss.normalize_L2(query_embedding)
            scores, indices = self.index.search(query_embedding, top_k)

            contexts = []
            for i, idx in enumerate(indices[0]):
                if idx < len(self.knowledge_base):
                    disease = self.knowledge_base[idx]["disease"]
                    symptoms = self.knowledge_base[idx]["symptoms"]
                    contexts.append(f"• {disease}: {symptoms}")

            return "\n".join(contexts) if contexts else ""

        except Exception as e:
            print(f"Retrieve error: {e}")
            return ""

    def generate_medical_response(self, symptoms):
        if not self.model_loaded:
            return "❌ System initializing..."

        try:
            # Get relevant medical context
            context = self.smart_retrieve(symptoms)

            # Create much better prompt
            if context:
                prompt = f"""As an experienced medical assistant, analyze these symptoms in detail:

Patient Symptoms: {symptoms}

Relevant Medical Information from database:
{context}

Provide a detailed analysis with:
1. Most likely conditions based on symptom match
2. Key differentiating factors between these conditions
3. General self-care advice
4. When to see a doctor

Remember to be specific and helpful while clarifying you're an AI assistant.

Detailed Medical Analysis:"""
            else:
                prompt = f"""As a medical assistant, analyze these symptoms:

Patient Symptoms: {symptoms}

Provide detailed medical insights and practical suggestions:

Medical Analysis:"""

            # Get device
            device = self.model.device

            # Tokenize
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024
            ).to(device)

            # Generate response
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=350,  # Increased for more detailed response
                    temperature=0.85,    # Slightly more creative
                    do_sample=True,
                    top_p=0.92,
                    pad_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1,
                    early_stopping=True
                )

            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract only the new generated part
            if prompt in response:
                response = response.split(prompt)[-1].strip()

            # Ensure meaningful response - if still generic, enhance it
            if not response or len(response) < 30:
                # Enhanced fallback using RAG data
                if context:
                    diseases = []
                    symptoms_list = []

                    for line in context.split("\n"):
                        if "•" in line:
                            parts = line.split(":")
                            if len(parts) >= 2:
                                disease = parts[0].replace("• ", "").strip()
                                symptom_text = parts[1].strip()
                                diseases.append(disease)
                                symptoms_list.append(f"{disease}: {symptom_text}")

                    if diseases:
                        enhanced_response = f"""Based on your symptoms "{symptoms}", here's a detailed analysis:

**Most Likely Conditions:**
{', '.join(diseases)}

**Symptom Analysis:**
{chr(10).join(symptoms_list)}

**Recommendations:**
• Stay hydrated and rest
• Monitor your symptoms
• Consult a doctor if symptoms persist beyond 48 hours
• Seek immediate care if you experience severe dehydration signs

*Note: This is AI analysis based on medical data. Please consult a healthcare professional for accurate diagnosis.*"""
                        response = enhanced_response

            return response

        except Exception as e:
            print(f"Generation error: {e}")
            # Enhanced fallback
            context = self.smart_retrieve(symptoms)
            if context:
                return f"""**Symptom Analysis for: "{symptoms}"**

Based on medical knowledge, your symptoms may indicate:
{context}

**Advice:**
• Monitor your condition closely
• Maintain proper hydration
• Rest and avoid strenuous activities
• Consult a doctor for proper evaluation

*AI-powered analysis - seek professional medical advice for diagnosis*"""
            else:
                return f"""I've analyzed your symptoms: "{symptoms}"

For gastrointestinal issues like diarrhea, consider:
• Hydration is crucial - drink ORS or electrolyte solutions
• Avoid dairy, spicy foods, and caffeine
• BRAT diet (Bananas, Rice, Applesauce, Toast) may help
• Monitor for dehydration signs (dark urine, dizziness)

Please consult a healthcare provider for persistent symptoms."""

    def one_click_setup(self, file_path, base_model_name="microsoft/DialoGPT-medium", hf_token=None):
        """One-click setup - everything in single call"""
        try:
            # Step 1: Load data
            status, _ = self.load_and_clean_dataset(file_path)
            if "❌" in status:
                yield status
                return

            # Step 2: Setup model
            yield "🔄 Setting up AI model..."
            model_status = self.setup_model(base_model_name=base_model_name, hf_token=hf_token)
            if "❌" in str(model_status):
                yield model_status
                return

            # Step 3: Setup RAG (most important part)
            yield "🔄 Building medical knowledge base..."
            rag_status = self.setup_rag()
            if "❌" in str(rag_status):
                yield rag_status
                return

            self.system_ready = True
            yield f"✅ System Ready!\n- {rag_status}\n- Model: Loaded successfully\n- You can now chat with the medical assistant!"

        except Exception as e:
            yield f"❌ Setup failed: {str(e)}"

# Optimized Interface
medical_ai = MedicalAIFull()

def handle_one_click(file_path, base_model, hf_token):
    """Handle one-click setup with progress updates"""
    for status in medical_ai.one_click_setup(file_path, base_model, hf_token):
        yield status

def handle_chat(symptoms):
    if not symptoms.strip():
        return "❌ Please describe your symptoms"

    response = medical_ai.generate_medical_response(symptoms)
    return response

def handle_emergency(symptoms):
    if not symptoms.strip():
        return "❌ Please describe symptoms"

    emergencies = [
        "chest pain", "difficulty breathing", "severe bleeding",
        "unconscious", "heart attack", "stroke", "can't breathe",
        "severe pain", "sudden weakness", "vision loss", "choking"
    ]

    symptoms_lower = symptoms.lower()
    emergency_found = any(emergency in symptoms_lower for emergency in emergencies)

    if emergency_found:
        return "🚨 EMERGENCY: Call 108/112 immediately! This could be life-threatening. Do not wait!"
    else:
        return "ℹ️ This doesn't appear to be an emergency, but consult a healthcare professional for proper diagnosis."

# Better Interface
with gr.Blocks(title="Medical AI Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 Smart Medical AI Assistant")
    gr.Markdown("### Get intelligent symptom analysis with medical knowledge base")

    with gr.Tab("🚀 Quick Setup"):
        gr.Markdown("**Upload your medical data CSV and start chatting!**")
        file_input = gr.File(label="Upload Medical CSV (columns: disease, symptoms)", file_types=[".csv"], type="filepath")
        model_input = gr.Textbox(label="AI Model", value="microsoft/DialoGPT-medium", placeholder="HuggingFace model name")
        token_input = gr.Textbox(label="HF Token (optional)", type="password")
        setup_btn = gr.Button("🎯 Initialize Medical AI", variant="primary")
        setup_status = gr.Textbox(label="Setup Progress", interactive=False, lines=4)

    with gr.Tab("💬 Medical Consultation"):
        gr.Markdown("### Describe your symptoms for analysis")
        symptoms_input = gr.Textbox(
            label="What symptoms are you experiencing?",
            placeholder="Example: headache, fever for 2 days, body pain, cough with phlegm...",
            lines=3
        )
        chat_btn = gr.Button("🔍 Analyze Symptoms", variant="primary")
        chat_output = gr.Textbox(
            label="Medical Analysis",
            interactive=False,
            lines=8,
            placeholder="Your analysis will appear here..."
        )

    with gr.Tab("⚠️ Emergency Check"):
        gr.Markdown("### Check if your symptoms require emergency care")
        emergency_input = gr.Textbox(
            label="Describe your symptoms",
            placeholder="Example: severe chest pain, difficulty breathing, sudden weakness...",
            lines=2
        )
        emergency_btn = gr.Button("🚨 Check Emergency", variant="stop")
        emergency_output = gr.Textbox(label="Emergency Assessment", interactive=False, lines=3)

    # Event handlers
    setup_btn.click(
        handle_one_click,
        [file_input, model_input, token_input],
        [setup_status]
    )

    chat_btn.click(
        handle_chat,
        [symptoms_input],
        [chat_output]
    )

    emergency_btn.click(handle_emergency, [emergency_input], [emergency_output])

if __name__ == "__main__":
    demo.launch(
        share=True,
        debug=False,
        show_error=True
    )

🚀 Medical AI Full-Pipeline Initialized


/tmp/ipython-input-2440746876.py:367: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Medical AI Assistant", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa8f0b901158078618.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
